<a href="https://colab.research.google.com/github/maxGrigorenko/DL_HSE/blob/hw_6/MHLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Григоренко Максим. Тестирование MHLA (Multi-Head Linear Attention) на своих данных

In [ ]:
!git clone -b main --single-branch https://github.com/DAGroup-PKU/MHLA
%cd MHLA/mhla_image_classification

!pip install -r requirements.txt

In [1]:
# Скачиваем ImageNette (разрешение оригинального ImageNet)
!wget https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
!tar -xf imagenette2.tgz

!ls imagenette2

--2026-06-07 17:02:21--  https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.212.46, 16.182.109.56, 16.15.183.67, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.212.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1557161267 (1.5G) [application/x-tar]
Saving to: ‘imagenette2.tgz’

imagenette2.tgz     100%[===================>]   1.45G  56.4MB/s    in 30s     

2026-06-07 17:02:51 (50.0 MB/s) - ‘imagenette2.tgz’ saved [1557161267/1557161267]

noisy_imagenette.csv  train  val


In [10]:
%cd /content/MHLA/mhla_image_classification

# Запускаем обучение с правильным именем модели
!WANDB_MODE=disabled PYTHONPATH=.:$PYTHONPATH torchrun \
  --nproc_per_node=1 \
  --nnodes=1 \
  --master_port=12345 \
  timm_train.py \
  --model deit_small_pla_1d_v6_6 \
  --data-dir /content/imagenette2 \
  --opt adamw \
  --lr 1e-3 \
  --weight-decay 0.05 \
  --sched cosine \
  --epochs 5 \
  --warmup-epochs 1 \
  --warmup-lr 1e-6 \
  --batch-size 64 \
  --drop-path 0.1 \
  --reprob 0.25 \
  --mixup 0.2 \
  --cutmix 1.0 \
  --smoothing 0.1 \
  --aa rand-m9-mstd0.5-inc1 \
  --workers 4 \
  --model-ema \
  --model-ema-decay 0.9996 \
  --color-jitter 0.4 \
  --experiment mhla_imagenette_poc \
  --sched-on-updates \
  --gp avg \
  --model-kwargs piece_size=4 transform=linear exp_sigma=1

/content/MHLA/mhla_image_classification
2026-06-07 17:15:30.090997: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
Training with a single process on 1 device (cuda).
22200424
Model deit_small_pla_1d_v6_6 created, param count:22200424
Data processing configuration for current model + dataset:
	input_size: (3, 224, 224)
	interpolation: bicubic
	mean: (0.485, 0.456, 0.406)
	std: (0.229, 0.224, 0.225)
	crop_pct: 0.875
	crop_mode: center
Created AdamW (adamw) optimizer: lr: 0.001, betas

DeiT-Small со стандартным квадратичным вниманием:

In [11]:
%cd /content/MHLA/mhla_image_classification

# Запускаем эталонный DeiT со стандартным Softmax-вниманием
!WANDB_MODE=disabled PYTHONPATH=.:$PYTHONPATH torchrun \
  --nproc_per_node=1 \
  --nnodes=1 \
  --master_port=12347 \
  timm_train.py \
  --model deit_small \
  --data-dir /content/imagenette2 \
  --opt adamw \
  --lr 1e-3 \
  --weight-decay 0.05 \
  --sched cosine \
  --epochs 5 \
  --warmup-epochs 1 \
  --warmup-lr 1e-6 \
  --batch-size 64 \
  --drop-path 0.1 \
  --reprob 0.25 \
  --mixup 0.2 \
  --cutmix 1.0 \
  --smoothing 0.1 \
  --aa rand-m9-mstd0.5-inc1 \
  --workers 4 \
  --model-ema \
  --model-ema-decay 0.9996 \
  --color-jitter 0.4 \
  --experiment deit_small_baseline_poc \
  --sched-on-updates \
  --gp avg

/content/MHLA/mhla_image_classification
2026-06-07 17:41:13.636675: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
Training with a single process on 1 device (cuda).
22050664
Model deit_small created, param count:22050664
Data processing configuration for current model + dataset:
	input_size: (3, 224, 224)
	interpolation: bicubic
	mean: (0.485, 0.456, 0.406)
	std: (0.229, 0.224, 0.225)
	crop_pct: 0.9
	crop_mode: center
Created AdamW (adamw) optimizer: lr: 0.001, betas: (0.9, 0.999)

### Интерпретация результатов
Сравнение моделей DeiT-Small (стандартное внимание, $O(N^2)$) и модификации MHLA (линейное внимание, $O(N)$) за 5 эпох на ImageNette показало:

Точность: MHLA превзошел базовую модель по метрике Top-1 (29.48% против 26.98%). Базовая модель показала незначительное преимущество в Top-5 (75.82% против 73.38%) и Train Loss (2.84 против 2.91).

Скорость: Базовая архитектура работала быстрее (80-85 ит/с против 50-51). На коротких последовательностях ($N=196$) алгоритмическое преимущество $O(N)$ нивелируется отсутствием низкоуровневых CUDA-оптимизаций у кастомного слоя MHLA.

###Выводы
Выразительность сохранена: MHLA успешно избегает "коллапса контекста", не уступая классическому вниманию в качестве извлечения признаков.

Интеграция MHLA целесообразна для задач c длинным контекстом, где классическое внимание $O(N^2)$ приводит к высокой вычислительной сложности и нехватке памяти .